# B2：动作、自由 rollout 与逐帧噪声

这一次比较动作怎样进入模型，让模型连续读取自己的输出，再为视频中的每一帧指定不同噪声等级。每次只换一个条件。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'hwm').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import torch
from hwm.data import make_pixelworld_dataset
from hwm.video import (
    TinyVQVAE, ActionTokenTransformer, TinyConditionalDenoiser,
    add_independent_frame_noise, diffusion_prediction_target,
    motion_direction_accuracy, red_centers, rollout_token_model,
    video_batch_from_episodes,
)
torch.manual_seed(1)


## 1. 训练一个很小的离散视频模型

这里重复 B1 的短训练，使 Notebook 可以单独运行。PA1-B 会保存 tokenizer，不会每次重训。

In [ ]:
episodes = make_pixelworld_dataset(12, 8, seed=2)
current, actions, following = video_batch_from_episodes(episodes)
images = torch.cat((current, following))
tokenizer = TinyVQVAE(codebook_size=16, embedding_size=8)
opt = torch.optim.Adam(tokenizer.parameters(), lr=1e-3)
for _ in range(30):
    opt.zero_grad(); loss, _ = tokenizer.continuous_loss(images); loss.backward(); opt.step()
tokenizer.initialize_codebook(images)
with torch.no_grad():
    current_tokens = tokenizer.encode_tokens(current).flatten(1); next_tokens = tokenizer.encode_tokens(following).flatten(1)
dynamics = ActionTokenTransformer(codebook_size=16, model_size=32)
opt = torch.optim.Adam(dynamics.parameters(), lr=3e-3)
for _ in range(40):
    opt.zero_grad(); loss = dynamics.loss(current_tokens, actions, next_tokens); loss.backward(); opt.step()
print('最后 token loss:', round(float(loss.detach()), 4))


## 2. 动作怎样进入模型

保持 token、模型宽度和数据不变，只切换 `none / additive / film`。`none` 是必要基线；additive 最简单；FiLM 用动作缩放和平移视觉特征。复杂方法不一定更好。

In [ ]:
same_start = current_tokens[:1].expand(5, -1)
all_actions = torch.arange(5)
models = {}
for injection in ('none', 'additive', 'film'):
    torch.manual_seed(7)
    model = ActionTokenTransformer(codebook_size=16, model_size=32, action_injection=injection)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    for _ in range(25):
        optimizer.zero_grad(); train_loss = model.loss(current_tokens, actions, next_tokens); train_loss.backward(); optimizer.step()
    with torch.no_grad():
        logits = model(same_start, all_actions)
        frames = tokenizer.decode_tokens(logits.argmax(-1).reshape(-1, 4, 4))
    sensitivity = (logits - logits[:1]).abs().mean().item()
    print(injection, 'loss=', round(float(train_loss.detach()), 3), 'sensitivity=', round(sensitivity, 4), 'centers=', red_centers(frames).tolist())
    models[injection] = model
dynamics = models['additive']
print('比较时只更换 action injection；logits 变化后仍要检查解码位置。')


## 3. Teacher forcing 与 free rollout

一步预测时输入来自真实数据；自由生成时，第二步开始输入来自模型自己。两者的差距正是许多 forcing 方法想解决的问题。

In [ ]:
with torch.no_grad():
    teacher_tokens = dynamics(current_tokens, actions).argmax(-1)
    teacher_frames = tokenizer.decode_tokens(teacher_tokens.reshape(-1, 4, 4))
teacher_direction = motion_direction_accuracy(current, teacher_frames, following)
right_actions = [torch.tensor([2]) for _ in range(8)]
rollout_frames = rollout_token_model(dynamics, tokenizer, current_tokens[:1], right_actions, (4, 4))
centers = red_centers(rollout_frames)
print('teacher-forced 一步方向准确率:', teacher_direction)
print('free rollout 连续向右时的解码中心:', centers)
print('中心停止、反向或消失，都是自由 rollout 暴露的失败。')


## 4. Diffusion Forcing 的最小接口

把 transition 整理成 `[B,T,C,H,W]`，让每一帧拥有自己的噪声等级：历史接近干净，较远未来更嘈杂。同一份 noisy video 可以配 `x / epsilon / v` 三种监督目标。

In [ ]:
clip = following[:12].reshape(4, 3, *following.shape[1:])
levels = torch.tensor([[0.0, 0.5, 1.0]]).expand(4, -1)
noisy_clip, sampled_noise = add_independent_frame_noise(clip, levels)
for target_name in ('x', 'epsilon', 'v'):
    target = diffusion_prediction_target(clip, sampled_noise, levels, target_name)
    print(target_name, 'target shape:', tuple(target.shape), 'std:', round(float(target.std()), 3))
denoiser = TinyConditionalDenoiser()
opt = torch.optim.Adam(denoiser.parameters(), lr=2e-3)
noise_level = torch.full((len(current),), 0.2)
noise = torch.randn_like(following)
noisy = (1-noise_level[:,None,None,None])*following + noise_level[:,None,None,None]*noise
losses = []
for _ in range(20):
    opt.zero_grad(); predicted = denoiser(noisy, current, actions, noise_level); loss = torch.nn.functional.mse_loss(predicted, noise); loss.backward(); opt.step(); losses.append(float(loss.detach()))
print('tiny epsilon loss:', round(losses[0], 3), '→', round(losses[-1], 3))
assert losses[-1] < losses[0]


## 小结

这一份 Notebook 对应三条研究主线：动作怎样进入模型、训练历史与自由 rollout 为何不同、逐帧噪声怎样统一下一帧预测与整段生成。PA1-B 必须一次只换一个轴，并同时报告动作、长时和延迟。